# Ordenação

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Acompanhar as intercalações do merge sort e explicar por que ele leva O(n log n)
- Escrever a etapa de intercalação e reaproveitá-la para contar inversões
- Ordenar por uma chave com um comparator e depois percorrer os dados ordenados numa passada
- Usar a ordenação por contagem quando os valores ficam num intervalo pequeno

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## Em resumo

Ordenar deixa os dados fáceis de buscar, agrupar e comparar, e as principais ordenações O(n log n) usam uma de duas ideias.

O **merge sort** intercala sequências ordenadas. Intercalar duas listas ordenadas custa uma comparação por valor retirado, então uma passada por n valores custa uns n. Comece com sequências de 1 e dobre o tamanho a cada passada: depois de log₂ n passadas sobra uma sequência ordenada, com O(n log n) no total, sempre.

#### 🎞️ Intercalando duas sequências ordenadas

![Intercalando 2 5 8 com 1 6 9: a cada vez sai o menor valor da frente, dando 1 2 5 6 8 9](merge-steps.pt-br.svg)

![As passadas do merge sort em 8 valores](passes.pt-br.svg)

**O(n²) contra O(n log n):** ordenação por seleção e `Arrays.sort` nos mesmos números aleatórios. Dobre `n` e compare.

In [ ]:
int n = 20_000;
int[] a = new Random(2).ints(n, 0, 1_000_000).toArray();
int[] b = a.clone();

long inicio = System.nanoTime();
for (int i = 0; i < n; i++) { // ordenação por seleção
    int min = i;
    for (int j = i + 1; j < n; j++) {
        if (a[j] < a[min]) {
            min = j;
        }
    }
    int t = a[i];
    a[i] = a[min];
    a[min] = t;
}
long meio = System.nanoTime();
Arrays.sort(b);
long fim = System.nanoTime();

IO.println("seleção:     " + (meio - inicio) / 1_000_000 + " ms");
IO.println("Arrays.sort: " + (fim - meio) / 1_000_000 + " ms");
IO.println("mesmo resultado: " + Arrays.equals(a, b));

### ✏️ Exercício: Intercale dois arrays ordenados

O passo no coração do merge sort: monte um array ordenado a partir de dois ordenados, pegando sempre o menor valor da frente.

In [ ]:
int[] merge(int[] a, int[] b) {
    // TODO: pegue o menor valor da frente até um lado acabar, depois copie o resto
    return new int[0];
}

void main() {
    check("misturados", Arrays.toString(merge(new int[] {2, 5, 8}, new int[] {1, 6, 9})), "[1, 2, 5, 6, 8, 9]");
    check("um vazio", Arrays.toString(merge(new int[] {}, new int[] {3, 4})), "[3, 4]");
    check("repetidos", Arrays.toString(merge(new int[] {1, 1}, new int[] {1})), "[1, 1, 1]");
}

<details>
<summary>Ver uma resposta</summary>

```java
int[] merge(int[] a, int[] b) {
    int[] out = new int[a.length + b.length];
    int i = 0, j = 0, k = 0;
    while (i < a.length && j < b.length) {
        out[k++] = a[i] <= b[j] ? a[i++] : b[j++];
    }
    while (i < a.length) {
        out[k++] = a[i++];
    }
    while (j < b.length) {
        out[k++] = b[j++];
    }
    return out;
}

void main() {
    check("misturados", Arrays.toString(merge(new int[] {2, 5, 8}, new int[] {1, 6, 9})), "[1, 2, 5, 6, 8, 9]");
    check("um vazio", Arrays.toString(merge(new int[] {}, new int[] {3, 4})), "[3, 4]");
    check("repetidos", Arrays.toString(merge(new int[] {1, 1}, new int[] {1})), "[1, 1, 1]");
}
```

</details>

O **quicksort** escolhe um pivô, move os valores menores para a esquerda dele e os maiores para a direita (uma **partição**), e ordena cada lado. Ele é O(n log n) em média, mas um pivô ruim toda vez, como o primeiro elemento de um array ordenado, o deixa O(n²).

**Um pivô ruim toda vez.** Este quicksort sempre escolhe o primeiro elemento. Conte as comparações em dados aleatórios e em dados já ordenados.

In [ ]:
// shared: quicksort com o primeiro elemento como pivô, contando comparações
long comparisons = 0;

void quicksort(int[] a, int lo, int hi) {
    if (lo >= hi) {
        return;
    }
    int pivo = a[lo];
    int i = lo;
    for (int j = lo + 1; j <= hi; j++) {
        comparisons++;
        if (a[j] < pivo) {
            i++;
            int t = a[i];
            a[i] = a[j];
            a[j] = t;
        }
    }
    int t = a[lo];
    a[lo] = a[i];
    a[i] = t;
    quicksort(a, lo, i - 1);
    quicksort(a, i + 1, hi);
}

In [ ]:
int n = 3000;
int[] aleatorio = new Random(3).ints(n, 0, 1_000_000).toArray();
quicksort(aleatorio, 0, n - 1);
IO.println(String.format("aleatório: %,d comparações", comparisons));

comparisons = 0;
int[] ordenado = new int[n];
for (int i = 0; i < n; i++) {
    ordenado[i] = i;
}
quicksort(ordenado, 0, n - 1);
IO.println(String.format("ordenado:  %,d comparações (n²/2 = %,d)", comparisons, (long) n * n / 2));

Quando os valores são inteiros pequenos, como notas de 0 a 100, a **ordenação por contagem** nem compara: ela conta quantas vezes cada valor aparece, em O(n + k) para k valores possíveis.

### ✏️ Exercício: Ordenação por inserção

Ordene o array no lugar: pegue cada valor e deslize-o para a esquerda, passando pelos valores maiores antes dele. É O(n²), mas muito rápido em dados quase ordenados.

In [ ]:
/** Sorts a copy with insertionSort and returns it as text. */
String sortedCopy(int[] a) {
    int[] copy = a.clone();
    insertionSort(copy);
    return Arrays.toString(copy);
}

void insertionSort(int[] a) {
    // TODO: para cada i, mova a[i] para a esquerda enquanto o valor antes dele for maior
}

void main() {
    check("ordena", sortedCopy(new int[] {5, 2, 9, 1}), "[1, 2, 5, 9]");
    check("repetidos", sortedCopy(new int[] {3, 1, 3, 2}), "[1, 2, 3, 3]");
    check("já ordenado", sortedCopy(new int[] {1, 2, 3}), "[1, 2, 3]");
}

<details>
<summary>Ver uma resposta</summary>

```java
/** Sorts a copy with insertionSort and returns it as text. */
String sortedCopy(int[] a) {
    int[] copy = a.clone();
    insertionSort(copy);
    return Arrays.toString(copy);
}

void insertionSort(int[] a) {
    for (int i = 1; i < a.length; i++) {
        int value = a[i];
        int j = i - 1;
        while (j >= 0 && a[j] > value) {
            a[j + 1] = a[j];
            j--;
        }
        a[j + 1] = value;
    }
}

void main() {
    check("ordena", sortedCopy(new int[] {5, 2, 9, 1}), "[1, 2, 5, 9]");
    check("repetidos", sortedCopy(new int[] {3, 1, 3, 2}), "[1, 2, 3, 3]");
    check("já ordenado", sortedCopy(new int[] {1, 2, 3}), "[1, 2, 3]");
}
```

</details>

Em Java, `Arrays.sort` num `int[]` usa um quicksort, e em objetos um merge sort guiado por um **comparator**, como `Comparator.comparingInt(Meeting::start)`. Essa ordenação é **estável**: elementos iguais mantêm a ordem em que estavam.

Cuidado: um comparator escrito como `a - b` pode estourar. Use `Integer.compare(a, b)`.

**Comparators:** pelo tamanho, depois em ordem alfabética, e invertido. A última linha mostra o estouro de `a - b`.

In [ ]:
List<String> palavras = new ArrayList<>(List.of("pear", "fig", "apple", "kiwi", "date"));
palavras.sort(Comparator.comparingInt(String::length).thenComparing(Comparator.naturalOrder()));
IO.println(palavras);
palavras.sort(Comparator.comparingInt(String::length).reversed());
IO.println(palavras);

List<Integer> numeros = new ArrayList<>(List.of(2_000_000_000, -1_000_000_000, 0));
numeros.sort((a, b) -> a - b); // estoura
IO.println("com a - b:            " + numeros);
numeros.sort(Integer::compare);
IO.println("com Integer.compare:  " + numeros);

## Leituras

- [Sorting](https://visualgo.net/en/sorting) (VisuAlgo (National University of Singapore), 20 min): Escolha Merge Sort, Quick Sort e Counting Sort no menu de cima e rode cada um num array curto seu. Repare na intercalação pegando o menor valor da frente, na partição movendo valores em volta do pivô e na ordenação por contagem nunca comparando dois valores.
- [Writing and Combining Comparators](https://dev.java/learn/language/fp/lambdas/writing-comparators/) (dev.java (Oracle), 8 min): Comparator.comparing e comparingInt montam um comparator a partir de uma chave, thenComparing desempata com uma segunda chave e reversed() inverte a ordem. Repare por que (i1 - i2) é um comparator ruim.

## Verifique

1. Quantas passadas o merge sort de baixo para cima faz em 1.000 valores?
2. Por que a ordenação por contagem não ordena nomes?
3. O que uma ordenação estável mantém, e quando isso importa?

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Notas da prova** · Fácil
- **Rastro do merge** · Médio
- **Agenda unificada** · Médio
- **Distância entre rankings** · Difícil